# Notebook 2 - Construccion y resolucion del modelo en Julia

Entrada esperada desde Notebook 1:
- `out/S.csv`
- `out/lb.csv`
- `out/ub.csv`
- `out/rxn_ids.txt`
- `out/met_ids.txt`

Este notebook arma el problema MPCC/pFBA y ejecuta la simulacion dFBA.


In [1]:
using JuMP
using Ipopt
using LinearAlgebra
using DelimitedFiles
using Printf
using Statistics
using Plots

include("pFBA_KKT_flux_Zenteno.jl")

OUT_DIR = get(ENV, "OUT_DIR", joinpath(pwd(), "out"))
S_FILE = joinpath(OUT_DIR, "S.csv")
LB_FILE = joinpath(OUT_DIR, "lb.csv")
UB_FILE = joinpath(OUT_DIR, "ub.csv")
RXN_FILE = joinpath(OUT_DIR, "rxn_ids.txt")
MET_FILE = joinpath(OUT_DIR, "met_ids.txt")
XK_FILE = joinpath(OUT_DIR, "xk.csv")
T_FILE = joinpath(OUT_DIR, "t.csv")
FIG_FILE = joinpath(OUT_DIR, "dfba.png")

for f in (S_FILE, LB_FILE, UB_FILE, RXN_FILE, MET_FILE)
    @assert isfile(f) "Falta archivo requerido: $(f). Ejecuta Notebook 1 primero."
end

println("Entradas encontradas en ", OUT_DIR)


Entradas encontradas en c:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\out


In [ ]:
S = Float64.(readdlm(S_FILE, ','))
lbraw = readdlm(LB_FILE, ',')
ubraw = readdlm(UB_FILE, ',')
RXN_IDS = readlines(RXN_FILE)
MET_IDS = readlines(MET_FILE)

vlb = lbraw isa AbstractVector ? Float64.(lbraw) : Float64.(lbraw[:, 1])
vub = ubraw isa AbstractVector ? Float64.(ubraw) : Float64.(ubraw[:, 1])
nm = size(S, 1)
nv = size(S, 2)

RXN_INDEX = Dict{String, Int}(rxn => i for (i, rxn) in pairs(RXN_IDS))
MET_INDEX = Dict{String, Int}(met => i for (i, met) in pairs(MET_IDS))

obj_id = if haskey(RXN_INDEX, "r_2111")
    "r_2111"
else
    error("No se encontro r_2111")
end

for rid in ["r_1761", "r_1714", "r_1709", "r_1992", "r_4046"]
    @assert haskey(RXN_INDEX, rid) "Falta reaccion clave: $(rid)"
end

eth = RXN_INDEX["r_1761"]
obj = RXN_INDEX[obj_id]
glu = RXN_INDEX["r_1714"]
fru = RXN_INDEX["r_1709"]
o2  = RXN_INDEX["r_1992"]
IDX_ATPM = RXN_INDEX["r_4046"]

KINETIC_N_SOURCE_IDS = [
    "r_1654", "r_1879", "r_1891", "r_1889",
    "r_1906", "r_1911", "r_1873", "r_1912",
]
for rid in KINETIC_N_SOURCE_IDS
    @assert haskey(RXN_INDEX, rid) "Falta fuente de N: $(rid)"
end

NITROGEN_SOURCES = [RXN_INDEX[rid] for rid in KINETIC_N_SOURCE_IDS]
UPTAKE_IDXS = vcat([glu, fru], NITROGEN_SOURCES)
n_up = length(UPTAKE_IDXS)

IS_GLU = [x == glu ? 1.0 : 0.0 for x in UPTAKE_IDXS]
IS_FRU = [x == fru ? 1.0 : 0.0 for x in UPTAKE_IDXS]
IS_NIT = [1.0 - IS_GLU[i] - IS_FRU[i] for i in 1:n_up]

idx_NH4 = RXN_INDEX["r_1654"]
idx_Arg = RXN_INDEX["r_1879"]
idx_Gln = RXN_INDEX["r_1891"]
idx_Glu_met = RXN_INDEX["r_1889"]
idx_Ser = RXN_INDEX["r_1906"]
idx_Thr = RXN_INDEX["r_1911"]
idx_Ala = RXN_INDEX["r_1873"]
idx_Trp = RXN_INDEX["r_1912"]

N_atoms_dict = Dict(
    idx_NH4 => 1.0,
    idx_Arg => 4.0,
    idx_Gln => 2.0,
    idx_Glu_met => 1.0,
    idx_Ser => 1.0,
    idx_Thr => 1.0,
    idx_Ala => 1.0,
    idx_Trp => 2.0,
)
N_profile_dict = Dict(
    idx_NH4 => 0.40,
    idx_Arg => 0.20,
    idx_Gln => 0.10,
    idx_Glu_met => 0.05,
    idx_Ser => 0.05,
    idx_Thr => 0.05,
    idx_Ala => 0.05,
    idx_Trp => 0.10,
)

MW_N   = 0.014007
MW_GLU = 0.180156
MW_FRU = 0.180156
MW_ETH = 0.046070
MW_O2  = 0.031998

SELECT_UPTAKE = [Float64(mc == UPTAKE_IDXS[k]) for mc in 1:nv, k in 1:n_up]
N_atoms_vec = ones(nv)
N_profile_vec = zeros(nv)
for k in keys(N_atoms_dict);   1 <= k <= nv && (N_atoms_vec[k] = N_atoms_dict[k]);   end
for k in keys(N_profile_dict); 1 <= k <= nv && (N_profile_vec[k] = N_profile_dict[k]); end

N_frac = zeros(n_up)
for k in 1:n_up
    mc = UPTAKE_IDXS[k]
    if IS_NIT[k] > 0.5
        N_frac[k] = N_profile_vec[mc] / (N_atoms_vec[mc] * MW_N)
    end
end

PRODUCT_IDXS = [eth, obj]
n_prod = length(PRODUCT_IDXS)
SELECT_PRODUCT = [Float64(mc == PRODUCT_IDXS[k]) for mc in 1:nv, k in 1:n_prod]
IS_ETH_prod = [1.0, 0.0]
IS_OBJ_prod = [0.0, 1.0]

println("Datos listos: nm=$(nm), nv=$(nv), obj=$(obj_id)")


In [ ]:
ZENTENO_PARAM_SET = lowercase(get(ENV, "ZENTENO_PARAM_SET", "legacy_calibrated"))
if ZENTENO_PARAM_SET == "paper2010"
    MU0_nom    = 0.18
    YXN_nom    = 19.69
    YXG_nom    = 1.60
    YXF_nom    = 1.60
    YEG_nom    = 0.49
    YEF_nom    = 0.49
    Kn0_nom    = 0.01
    Kg0_nom    = 7.5
    Kf0_nom    = 7.5
    Kig0_nom   = 55.0
    Kie0_nom   = 40.0
    Kd0_nom    = 0.00044
    betaG0_nom = 0.225
    betaF0_nom = 0.225
elseif ZENTENO_PARAM_SET == "legacy_calibrated"
    MU0_nom    = 0.141665
    YXN_nom    = 9.80576
    YXG_nom    = 0.394345
    YXF_nom    = 0.18622
    YEG_nom    = 0.14133
    YEF_nom    = 0.96932
    Kn0_nom    = 0.226882
    Kg0_nom    = 3.1514
    Kf0_nom    = 2.97625
    Kig0_nom   = 29.5276
    Kie0_nom   = 2.99809
    Kd0_nom    = 0.0000311736
    betaG0_nom = 1.41182
    betaF0_nom = 8.49482
else
    error("ZENTENO_PARAM_SET invalido: $(ZENTENO_PARAM_SET)")
end

MU0 = MU0_nom
YEG = YEG_nom
YEF = YEF_nom
YXN = YXN_nom
MRATE_0 = 0.01

R_GAS = 8.314
R = R_GAS
EPS = 1e-9

T_BASE = try parse(Float64, get(ENV, "T_CONST", "293.15")) catch; 293.15 end
T_STEPS = [36.0, 96.0]
T_DELTAS = [5.0, 3.0]
T_STEEP = 0.5

function dynamic_temperature(t)
    val = T_BASE
    for idx in eachindex(T_STEPS)
        sigmoid = 1.0 / (1.0 + exp(-T_STEEP * (t - T_STEPS[idx])))
        val += T_DELTAS[idx] * sigmoid
    end
    return val
end

function death_rate_T(E, T_val)
    Td = -0.0001 * E^3 + 0.0049 * E^2 - 0.1279 * E + 315.89
    s = 0.5 * (1.0 + tanh(0.5 * (T_val - Td)))
    base = Kd0_nom * exp(0.0415 * E + (130000.0 * (T_val - 305.65)) / (305.65 * R * T_val))
    return base * s
end

SQRT_2PI = sqrt(2.0 * pi)
function smooth_injection(t, t_shot, dose, width=1.0)
    abs(t - t_shot) > 5 * width && return 0.0
    return (dose / (width * SQRT_2PI)) * exp(-0.5 * ((t - t_shot) / width)^2)
end

T_INJ_1 = 0.0; DOSE_1 = 0.0; WIDTH_1 = 5.0
T_INJ_2 = 0.0; DOSE_2 = 0.0; WIDTH_2 = 5.0

println("Parametros cineticos cargados: ", ZENTENO_PARAM_SET)
print("MU0=$(MU0), YXN=$(YXN), YXG=$(YXG_nom), YXF=$(YXF_nom), YEG=$(YEG), YEF=$(YEF), Kn0=$(Kn0_nom), Kg0=$(Kg0_nom), Kf0=$(Kf0_nom), Kig0=$(Kig0_nom), Kie0=$(Kie0_nom), Kd0=$(Kd0_nom), betaG0=$(betaG0_nom), betaF0=$(betaF0_nom)")


In [ ]:
O2_SAT = 0.26
EPS_FLUX = try parse(Float64, get(ENV, "EPS_FLUX", "1e-6")) catch; 1e-6 end
O2_ZERO_TOL = try parse(Float64, get(ENV, "O2_ZERO_TOL", "1e-7")) catch; 1e-7 end
LINEAR_SOLVER = lowercase(get(ENV, "IPOPT_LINEAR_SOLVER", "mumps"))

# Misma idea que Notebook 1: sin capeo de produccion/biomasa por defecto
APPLY_PRODUCT_CAPS = lowercase(get(ENV, "APPLY_PRODUCT_CAPS", "false")) in ("1", "true", "yes", "y", "on")

nc = 6
d  = zeros(nv); d[obj] = -1.0
w = 1e-20
phi1 = 1.0
phi2 = 1.0
phi3 = 1.0
phi4 = 1.0

cs = [1.0, 0.2, 100.0, 100.0, 10.0, 0.01]
FLUX_SCALE_TARGET = 50.0
vs = ones(nv)
n_scaled = 0
for mc in 1:nv
    bound_range = max(abs(vlb[mc]), abs(vub[mc]))
    if bound_range > FLUX_SCALE_TARGET
        vs[mc] = bound_range / FLUX_SCALE_TARGET
        n_scaled += 1
    end
end

X0 = 0.5
N0 = 0.14
G0 = 110.0
F0 = 110.0
E0 = 0.0
O2_0 = 0.0
c0 = [X0, N0, G0, F0, E0, O2_0]

nfe = 18
ncp = 3
th = 168.0
h = th / nfe
hm = fill(h, nfe)
var_h = 0.5

println("MPCC configurado: nfe=$(nfe), ncp=$(ncp), th=$(th)h, scaled=$(n_scaled)/$(nv)")
println("CAPEOS PRODUCCION/BIOMASA: ", APPLY_PRODUCT_CAPS ? "ON" : "OFF (sin capeo)")


In [ ]:
# PRECHECK: factibilidad estructural de restricciones (antes de optimizar)

function _kinetic_limits_at(c_phys::AbstractVector{<:Real}, t::Real)
    cX, cN, cG, cF, cE, _cO2 = c_phys

    T_val  = dynamic_temperature(t)
    mu_T   = exp(59453.0 * (T_val - 300.0) / (300.0 * R * T_val))
    Kg_T   = exp(46055.0 * (T_val - 293.15) / (293.15 * R * T_val))
    b_T    = exp(11000.0 * (T_val - 296.15) / (296.15 * R * T_val))
    mrate  = MRATE_0 * exp(37681.0 * (T_val - 293.30) / (293.30 * R * T_val))

    mu     = MU0 * mu_T * (cN / (cN + Kn0_nom * Kg_T + EPS))
    betaG  = betaG0_nom * b_T *
             (cG / (cG + Kg0_nom * Kg_T + EPS)) *
             (Kie0_nom * Kg_T / (cE + Kie0_nom * Kg_T + EPS))
    betaF  = betaF0_nom * b_T *
             (cF / (cF + Kf0_nom * Kg_T + EPS)) *
             (Kig0_nom * Kg_T / (cG + Kig0_nom * Kg_T + EPS)) *
             (Kie0_nom * Kg_T / (cE + Kie0_nom * Kg_T + EPS))

    vx = mu
    vg = mu / YXG_nom + betaG / YEG + mrate * (cG / (cG + cF + EPS))
    vf = mu / YXF_nom + betaF / YEF + mrate * (cF / (cG + cF + EPS))
    vn = mu / YXN
    ve = (betaG + betaF) / MW_ETH

    L_upt = [IS_GLU[k] * (vg / MW_GLU) +
             IS_FRU[k] * (vf / MW_FRU) +
             IS_NIT[k] * vn * N_frac[k] for k in 1:n_up]

    L_prod = [IS_ETH_prod[k] * ve + IS_OBJ_prod[k] * vx for k in 1:n_prod]

    return (t=t, T=T_val, vx=vx, vg=vg, vf=vf, vn=vn, ve=ve, L_upt=L_upt, L_prod=L_prod)
end

# Preview temporal usando hm (sin hv optimizado)
tfe_preview = cumsum(hm)
diag_preview = [_kinetic_limits_at(c0, t) for t in tfe_preview]  # c0 fijo

idx_glu_upt = something(findfirst(==(glu), UPTAKE_IDXS))
idx_fru_upt = something(findfirst(==(fru), UPTAKE_IDXS))
idx_obj_prod = something(findfirst(==(obj), PRODUCT_IDXS))
idx_eth_prod = something(findfirst(==(eth), PRODUCT_IDXS))
idx_nit_upt = findall(k -> IS_NIT[k] > 0.5, 1:n_up)
@assert !isempty(idx_nit_upt) "No se detectaron fuentes de N en UPTAKE_IDXS"

function _check_uptake(diag_preview, idx_rxn, idx_k)
    min_margin = Inf
    worst_i = 0
    for i in eachindex(diag_preview)
        L = diag_preview[i].L_upt[idx_k]
        lb, ub = vlb[idx_rxn], vub[idx_rxn]
        margin = ub - max(lb, -L)  # -v <= L
        if margin < min_margin
            min_margin = margin
            worst_i = i
        end
    end
    return min_margin, worst_i
end

function _check_product(diag_preview, idx_rxn, idx_k)
    min_margin = Inf
    worst_i = 0
    for i in eachindex(diag_preview)
        L = diag_preview[i].L_prod[idx_k]
        lb, ub = vlb[idx_rxn], vub[idx_rxn]
        margin = min(ub, L) - lb  # v <= L
        if margin < min_margin
            min_margin = margin
            worst_i = i
        end
    end
    return min_margin, worst_i
end

m_glu, i_glu = _check_uptake(diag_preview, glu, idx_glu_upt)
m_fru, i_fru = _check_uptake(diag_preview, fru, idx_fru_upt)
m_obj, i_obj = _check_product(diag_preview, obj, idx_obj_prod)
m_eth, i_eth = _check_product(diag_preview, eth, idx_eth_prod)

nit_margins = [_check_uptake(diag_preview, UPTAKE_IDXS[k], k)[1] for k in idx_nit_upt]
m_nit_src = minimum(nit_margins)
k_nit_worst = idx_nit_upt[argmin(nit_margins)]
rxn_nit_worst = UPTAKE_IDXS[k_nit_worst]

Lglu = [d.L_upt[idx_glu_upt] for d in diag_preview]
Lfru = [d.L_upt[idx_fru_upt] for d in diag_preview]
Lobj = [d.L_prod[idx_obj_prod] for d in diag_preview]
Leth = [d.L_prod[idx_eth_prod] for d in diag_preview]

# Limite agregado de N (gN/gDW/h), consistente con la figura final
limN_preview = [sum(max(0.0, d.L_upt[k]) * N_atoms_vec[UPTAKE_IDXS[k]] * MW_N for k in idx_nit_upt)
                for d in diag_preview]

# Capacidad maxima efectiva de N (kinetica + bound inferior de flujo)
capN_preview = [sum(max(0.0, min(-vlb[UPTAKE_IDXS[k]], d.L_upt[k])) * N_atoms_vec[UPTAKE_IDXS[k]] * MW_N
                    for k in idx_nit_upt)
                for d in diag_preview]

println("\n", "="^72)
println("[PRECHECK] Factibilidad estructural antes de optimizar")
println("="^72)
println("Escalas vs (interno -> físico):")
println(@sprintf("  glu: %.6g | fru: %.6g | obj: %.6g | eth: %.6g | o2: %.6g",
    vs[glu], vs[fru], vs[obj], vs[eth], vs[o2]))
println(@sprintf("EPS_FLUX interno=%.3e -> físico aprox: glu=%.3e, fru=%.3e, obj=%.3e",
    EPS_FLUX, EPS_FLUX*vs[glu], EPS_FLUX*vs[fru], EPS_FLUX*vs[obj]))

println("\nMárgenes algebraicos (>=0 factible):")
println(@sprintf("  uptake glu:  %+10.4e (FE=%d)", m_glu, i_glu))
println(@sprintf("  uptake fru:  %+10.4e (FE=%d)", m_fru, i_fru))
println(@sprintf("  prod obj:    %+10.4e (FE=%d)", m_obj, i_obj))
println(@sprintf("  prod eth:    %+10.4e (FE=%d)", m_eth, i_eth))
println(@sprintf("  uptake N (peor fuente): %+10.4e (rxn=%d)", m_nit_src, rxn_nit_worst))

println("\nRango límites preview (c0 fijo):")
println(@sprintf("  lim_glu: [%.4e, %.4e]", minimum(Lglu), maximum(Lglu)))
println(@sprintf("  lim_fru: [%.4e, %.4e]", minimum(Lfru), maximum(Lfru)))
println(@sprintf("  lim_obj: [%.4e, %.4e]", minimum(Lobj), maximum(Lobj)))
println(@sprintf("  lim_eth: [%.4e, %.4e]", minimum(Leth), maximum(Leth)))
println(@sprintf("  limN agregado [gN/gDW/h]: [%.4e, %.4e]", minimum(limN_preview), maximum(limN_preview)))
println(@sprintf("  capN (limite+bound)      : [%.4e, %.4e]", minimum(capN_preview), maximum(capN_preview)))
println("="^72)


In [ ]:
println("Resolviendo MPCC...")
println("apply_product_caps = ", APPLY_PRODUCT_CAPS)
sol, solv, solcd, soll, solalL, solalU, solag, solap, solh, diag = pFBA_KKT_flux_Zenteno_O2minimal(
    c0;
    eps_flux = EPS_FLUX,
    apply_product_caps = APPLY_PRODUCT_CAPS,
)

if :solver in keys(diag)
    println("solver term = ", diag.solver.term)
    println("primal = ", diag.solver.primal, " | dual = ", diag.solver.dual)
    println(@sprintf("max violaciones: uptake=%.3e, product=%.3e, max|v_o2|=%.3e",
            diag.max_uptake_violation, diag.max_product_violation, diag.o2_abs_max))
end


In [ ]:
radau_pts = [0.15505, 0.64495, 1.00000]

ts = Vector{Float64}(undef, nfe + 1)
ts[1] = 0.0
for i in 2:nfe+1
    ts[i] = ts[i-1] + solh[i-1]
end

n_pts = nfe * ncp + 1
tsn = Vector{Float64}(undef, n_pts)
xk = Matrix{Float64}(undef, nc, n_pts)

tsn[1] = 0.0
xk[:, 1] = [X0, N0, G0, F0, E0, O2_0]
for i in 1:nfe
    for j in 1:ncp
        kk = (i - 1) * ncp + j + 1
        xk[:, kk] = sol[:, i, j]
        tsn[kk] = ts[i] + radau_pts[j] * solh[i]
    end
end

writedlm(XK_FILE, xk)
writedlm(T_FILE, tsn)

println("Resultados guardados")
println("- ", XK_FILE)
println("- ", T_FILE)


In [ ]:
println()
println("="^60)
println("RESUMEN DE ESTADOS")
println("="^60)
for (s, name) in enumerate(("X [gDW/L]", "N [gN/L]", "G [g/L]", "F [g/L]", "E [g/L]", "O2 [g/L]"))
    vals = xk[s, :]
    println(@sprintf("%-12s: inicio=%8.4f  fin=%8.4f  min=%8.4f  max=%8.4f",
            name, vals[1], vals[end], minimum(vals), maximum(vals)))
end
println("hv range: ", minimum(solh), " -> ", maximum(solh), " h")

println()
println("Flujos clave (ultimo FE):")
println(@sprintf("v_obj (biomasa)  = %10.6f 1/h", solv[obj, nfe]))
println(@sprintf("v_glu (glucosa)  = %10.4f mmol/gDW/h", solv[glu, nfe]))
println(@sprintf("v_fru (fructosa) = %10.4f mmol/gDW/h", solv[fru, nfe]))
println(@sprintf("v_eth (etanol)   = %10.4f mmol/gDW/h", solv[eth, nfe]))
v_o2_last = abs(solv[o2, nfe]) <= O2_ZERO_TOL ? 0.0 : solv[o2, nfe]
println(@sprintf("v_o2 (oxigeno)   = %10.4f mmol/gDW/h", v_o2_last))
println("="^60)


In [ ]:
idx_glu_upt = findfirst(==(glu), UPTAKE_IDXS)
idx_fru_upt = findfirst(==(fru), UPTAKE_IDXS)
idx_obj_prod = findfirst(==(obj), PRODUCT_IDXS)
idx_eth_prod = findfirst(==(eth), PRODUCT_IDXS)
@assert idx_glu_upt !== nothing
@assert idx_fru_upt !== nothing
@assert idx_obj_prod !== nothing
@assert idx_eth_prod !== nothing
idx_glu_upt = something(idx_glu_upt)
idx_fru_upt = something(idx_fru_upt)
idx_obj_prod = something(idx_obj_prod)
idx_eth_prod = something(idx_eth_prod)

tfe = ts[2:end]
vX_fe = max.(0.0, solv[obj, :]); limX_fe = max.(0.0, diag.L_prod[idx_obj_prod, :])
vG_fe = max.(0.0, -solv[glu, :]); limG_fe = max.(0.0, diag.L_upt[idx_glu_upt, :])
vF_fe = max.(0.0, -solv[fru, :]); limF_fe = max.(0.0, diag.L_upt[idx_fru_upt, :])
vE_fe = max.(0.0, solv[eth, :]); limE_fe = max.(0.0, diag.L_prod[idx_eth_prod, :])
vO_fe = [abs(solv[o2, i]) <= O2_ZERO_TOL ? 0.0 : max(0.0, -solv[o2, i]) for i in 1:nfe]

vN_fe = zeros(nfe); limN_fe = zeros(nfe)
for i in 1:nfe
    for k in 1:n_up
        if IS_NIT[k] > 0.5
            idx = UPTAKE_IDXS[k]
            vN_fe[i] += max(0.0, -solv[idx, i]) * N_atoms_vec[idx] * MW_N
            limN_fe[i] += max(0.0, diag.L_upt[k, i]) * N_atoms_vec[idx] * MW_N
        end
    end
end

# Diagnostico de restricciones activas (flow/limit)
EPS_RATIO = 1e-12
rho_glu = vG_fe ./ max.(limG_fe, EPS_RATIO)
rho_fru = vF_fe ./ max.(limF_fe, EPS_RATIO)
rho_nit = vN_fe ./ max.(limN_fe, EPS_RATIO)

function ratio_label(rmax)
    if rmax > 1.02
        return "VIOLACION"
    elseif rmax >= 0.90
        return "APRETADO"
    elseif rmax >= 0.30
        return "INTERMEDIO"
    else
        return "HOLGADO"
    end
end

function print_ratio_summary(name, rho)
    rmin = minimum(rho)
    rmean = mean(rho)
    rmax = maximum(rho)
    label = ratio_label(rmax)
    println(@sprintf("  %-8s rho=min %.3f | mean %.3f | max %.3f -> %s", name, rmin, rmean, rmax, label))
end

println()
println("="^72)
println("ACTIVACION DE RESTRICCIONES (despues de optimizar)")
println("rho = flujo / limite  (cerca de 1 => restriccion activa)")
print_ratio_summary("glu", rho_glu)
print_ratio_summary("fru", rho_fru)
print_ratio_summary("N", rho_nit)
println("="^72)

pXc = plot(tsn, xk[1, :], label="X (gDW/L)", lw=2, color=:blue, xlabel="Tiempo (h)", ylabel="Concentracion")
pNc = plot(tsn, xk[2, :], label="N (gN/L)", lw=2, color=:green, xlabel="Tiempo (h)", ylabel="Concentracion")
pGc = plot(tsn, xk[3, :], label="G (g/L)", lw=2, color=:red, xlabel="Tiempo (h)", ylabel="Concentracion")
pFc = plot(tsn, xk[4, :], label="F (g/L)", lw=2, color=:orange, xlabel="Tiempo (h)", ylabel="Concentracion")
pEc = plot(tsn, xk[5, :], label="E (g/L)", lw=2, color=:purple, xlabel="Tiempo (h)", ylabel="Concentracion")
pOc = plot(tsn, xk[6, :], label="O2 (g/L)", lw=2, color=:cyan, xlabel="Tiempo (h)", ylabel="Concentracion")

pXv = plot(tfe, vX_fe, label="v_obj (1/h)", lw=2, color=:blue, xlabel="Tiempo (h)", ylabel="Flujo/limite")
plot!(pXv, tfe, limX_fe, label="lim_obj (1/h)", lw=2, color=:black, ls=:dash)
pNv = plot(tfe, vN_fe, label="vN efectivo", lw=2, color=:green, xlabel="Tiempo (h)", ylabel="Flujo/limite")
plot!(pNv, tfe, limN_fe, label="limN", lw=2, color=:black, ls=:dash)
pGv = plot(tfe, vG_fe, label="-v_glu", lw=2, color=:red, xlabel="Tiempo (h)", ylabel="Flujo/limite")
plot!(pGv, tfe, limG_fe, label="lim_glu", lw=2, color=:black, ls=:dash)
pFv = plot(tfe, vF_fe, label="-v_fru", lw=2, color=:orange, xlabel="Tiempo (h)", ylabel="Flujo/limite")
plot!(pFv, tfe, limF_fe, label="lim_fru", lw=2, color=:black, ls=:dash)
pEv = plot(tfe, vE_fe, label="v_eth", lw=2, color=:purple, xlabel="Tiempo (h)", ylabel="Flujo/limite")
plot!(pEv, tfe, limE_fe, label="lim_eth", lw=2, color=:black, ls=:dash)
pOv = plot(tfe, vO_fe, label="-v_o2", lw=2, color=:cyan, xlabel="Tiempo (h)", ylabel="Flujo")
plot!(pOv, tfe, zeros(nfe), label="lim_o2=0", lw=1.5, color=:black, ls=:dot)

plt = plot(pXc, pXv, pNc, pNv, pGc, pGv, pFc, pFv, pEc, pEv, pOc, pOv,
           layout=(6, 2), size=(1400, 1300), title="dFBA Zenteno (COBRA -> Julia)")
savefig(plt, FIG_FILE)
println("Grafico guardado: ", FIG_FILE)
display(plt)
